# 從 PyTorch 到 MNIST

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/02-From-PyTorch-to-MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_softmax.py', 'q_mlp.py', 'q_cnn.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from IPython.display import display
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 讀取 MNIST

`torchvision.datasets` 內建很多常用資料集。這裡我們直接把整個 MNIST 讀成 tensor（只有 60000x28x28，整個放進 GPU 也沒問題）。

In [ ]:
from torchvision.datasets import MNIST
train_ds = MNIST("data", train=True, download=True)
test_ds = MNIST("data", train=False, download=True)
train_X = train_ds.data.float().reshape(-1, 784) / 255
train_y = train_ds.targets
test_X = test_ds.data.float().reshape(-1, 784) / 255
test_y = test_ds.targets
train_X.shape, train_y.shape, test_X.shape

In [ ]:
def showX(X):
    int_X = (X * 255).clamp(0, 255).to(torch.uint8).cpu().numpy()
    int_X_reshape = int_X.reshape(-1, 28, 28).swapaxes(0, 1).reshape(28, -1)
    display(Image.fromarray(int_X_reshape))
showX(train_X[:20])
print(train_y[:20])

## Logistic Regression：從零開始

模型：$\hat y = \mathrm{softmax}(XW + b)$

Loss: cross entropy

In [ ]:
W = torch.zeros(784, 10, requires_grad=True)
b = torch.zeros(10, requires_grad=True)
def model(X):
    return X @ W + b     # 輸出的是 logits，還沒經過 softmax

## Q
自己寫 softmax 和 cross entropy，並與 `F.cross_entropy` 的結果比較。
* $\mathrm{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$
* cross entropy $= -\log \mathrm{softmax}(z)_{y}$ 的平均
* 提示：數值穩定的寫法要先減掉最大值；或直接用 `torch.logsumexp`

In [ ]:
# %load q_softmax.py
%run -i q_softmax.py

### 手寫訓練迴圈（mini-batch SGD）

In [ ]:
lr = 0.5
batch_size = 256
for epoch in range(5):
    perm = torch.randperm(len(train_X))
    for i in range(0, len(train_X), batch_size):
        idx = perm[i:i+batch_size]
        loss = F.cross_entropy(model(train_X[idx]), train_y[idx])
        loss.backward()
        with torch.no_grad():
            W -= lr * W.grad
            b -= lr * b.grad
            W.grad.zero_(); b.grad.zero_()
    with torch.no_grad():
        acc = (model(test_X).argmax(dim=1) == test_y).float().mean()
    print(f"epoch {epoch} loss={loss.item():.4f} test acc={acc:.4f}")

In [ ]:
# 看看 W 長什麼樣子
showX((W.detach().T - W.min()) / (W.max() - W.min()))

## 用 `nn.Module` 和 `torch.optim` 改寫

真正寫程式時，會用 `nn.Linear` 等現成的 layer，用 `DataLoader` 取 batch，用 optimizer 更新參數。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
train_loader = DataLoader(TensorDataset(train_X, train_y), batch_size=256, shuffle=True)
test_loader = DataLoader(TensorDataset(test_X, test_y), batch_size=1024)

In [ ]:
def train(model, epochs=5, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            loss = F.cross_entropy(model(X), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"epoch {epoch} loss={loss.item():.4f} test acc={evaluate(model):.4f}")

@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for X, y in test_loader:
        X, y = X.to(device), y.to(device)
        correct += (model(X).argmax(dim=1) == y).sum().item()
    return correct / len(test_loader.dataset)

In [ ]:
model = nn.Linear(784, 10)
train(model)

In [ ]:
# 預測看看 test_X 前 20 筆
with torch.no_grad():
    print(model(test_X[:20].to(device)).argmax(dim=1))
showX(test_X[:20])

## Q
建立一個 Multi-Layer Perceptron：
* `Linear(784, 256)` → `ReLU` → `Linear(256, 10)`
* 用 `nn.Sequential`
* 試試看加上 `nn.Dropout`、增加層數

In [ ]:
# %load q_mlp.py
%run -i q_mlp.py

### 自訂 `nn.Module`

複雜一點的模型通常寫成 class，在 `forward` 裡描述計算。

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden=512):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, 10)
    def forward(self, x):
        x = F.gelu(self.fc1(x))
        x = F.gelu(self.fc2(x)) + x   # residual connection
        return self.fc3(x)
model = MLP()
print(model)
print("參數數量:", sum(p.numel() for p in model.parameters()))

In [ ]:
train(model)

## Q：Convolutional Neural Network
建立以下的 CNN 架構，然後訓練看看
* reshape 成 `(N, 1, 28, 28)`（PyTorch 是 channel first）
* `Conv2d(1, 32, kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`
* `Conv2d(32, 64, kernel_size=3, padding=1)` → `ReLU` → `MaxPool2d(2)`
* `Flatten` → `Linear(64*7*7, 1024)` → `ReLU` → `Dropout(0.5)` → `Linear(1024, 10)`

提示：`nn.Unflatten(1, (1, 28, 28))` 可以放在 Sequential 的第一層。準確率應該可以到 99% 以上。

In [ ]:
# %load q_cnn.py
%run -i q_cnn.py

## 存檔與讀取

建議存 `state_dict`（只有參數），而不是整個 model 物件。

In [ ]:
torch.save(model.state_dict(), "mnist_cnn.pt")
model2 = cnn_model() if "cnn_model" in globals() else MLP()
model2.load_state_dict(torch.load("mnist_cnn.pt"))
model2.to(device)
evaluate(model2)